# 🛠️ 10.3 Setting Up PySpark

Welcome back! In the last lesson (**10.2 Spark Architecture**), we learned about the Driver, the Executors, and the Cluster Manager. 

Now, the theory is over. It is time to turn the ignition key and start the engine.

In this lesson, we are going to install Apache Spark on your local machine, initialize a **SparkSession** (the Driver), and write our very first PySpark program to process data.

### 🎯 Learning Objectives
* Understand how to install PySpark locally for development.
* Learn how to initialize a `SparkSession`.
* Write your first PySpark program to create a distributed DataFrame.
* Read a dataset (CSV) into PySpark.
* Compare how Data Engineers set up Spark environments versus other data roles.

## 1. Local Installation (Simulating a Cluster)

Wait a minute... didn't we say Spark was meant to run on a massive cluster of 1,000 computers? How can we run it on a single laptop?

Spark is incredibly flexible. When you run it on your laptop, Spark enters **"Local Mode"**. 
Instead of assigning tasks to 1,000 different computers, it treats the physical CPU cores inside your laptop as "Worker Nodes". If your laptop has 4 CPU cores, Spark simulates a miniature cluster of 4 Executors right inside your machine!

<img src="../assets/Module_10/10_03_01.png" alt="A visual showing a laptop split open. Inside the laptop screen is a miniature 'Driver' controlling several small 'Executors' running on the laptop's CPU chips." width="75%">

To get started in Python, we just need to install the PySpark library using `pip`.

In [ ]:
# Run this cell to install PySpark on your machine.
# (The '-q' flag just makes the installation output 'quiet' so it doesn't clutter our screen).

!pip install pyspark -q
print("✅ PySpark successfully installed!")

## 2. The Spark Session (Turning the Key)

As we learned in the architecture lesson, you must create a **SparkSession** to talk to the cluster. This is your Entry Point. It creates the **Driver**.

Let's break down the code we are about to run:
* `SparkSession.builder`: Starts the process of configuring our engine.
* `.appName("My First App")`: Gives our application a name (useful for tracking it in logs).
* `.master("local[*]")`: This tells Spark *where* to run. `local` means "run on my laptop". The `[*]` means "use every single CPU core available on this laptop to simulate executors."
* `.getOrCreate()`: Starts the engine. If one is already running, it just connects to it.

In [ ]:
# Importing the SparkSession module
from pyspark.sql import SparkSession

# 1. Initialize the SparkSession (This might take a few seconds to boot up)
print("Starting Spark engine...")
spark = SparkSession.builder \
    .appName("MyFirstSparkApp") \
    .master("local[*]") \
    .getOrCreate()

# 2. Verify it's running
print(f"✅ Spark engine started successfully!")
print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.conf.get('spark.app.name')}")

## 3. Your First PySpark Program

Now that the engine is running, let's process some data! 

In standard Python, you use lists or Pandas DataFrames. In PySpark, we use **Spark DataFrames**. A Spark DataFrame looks and acts like a normal table, but secretly, it is distributed across your cluster (or in our case, across your CPU cores).

Let's manually create a tiny Spark DataFrame.

In [ ]:
# 1. Create raw Python data (a list of tuples)
raw_data = [
    ("Alice", 28, "Data Engineer"),
    ("Bob", 35, "Data Scientist"),
    ("Charlie", 42, "Database Admin")
]

# 2. Define the column names
columns = ["Name", "Age", "Role"]

# 3. Convert the Python data into a Distributed Spark DataFrame
df = spark.createDataFrame(data=raw_data, schema=columns)

# 4. Show the DataFrame
print("My First Spark DataFrame:")
df.show()

## 4. Reading Your First Dataset

Manually typing data is fine for testing, but as a Data Engineer, you will be reading massive files from Cloud Storage or HDFS.

Spark makes this incredibly easy with the `spark.read` API. It can natively read CSV, JSON, Parquet, and database connections.

Let's write a quick script to generate a dummy CSV file on your machine, and then use Spark to read it.

In [ ]:
import os

# Step A: Create a dummy CSV file for us to read
csv_content = """transaction_id,amount,city
1001,50.00,New York
1002,120.50,London
1003,15.99,Tokyo
1004,200.00,New York"""

with open("sales_data.csv", "w") as file:
    file.write(csv_content)
print("Created 'sales_data.csv' on local disk.\n")

# Step B: Use PySpark to read the CSV file
# Notice how we pass options: header=True tells Spark the first row is column names,
# and inferSchema=True tells Spark to automatically guess if a column is a string or number.

print("Telling Spark to read the CSV file...")
sales_df = spark.read.csv(
    "sales_data.csv", 
    header=True, 
    inferSchema=True
)

# Display the data
sales_df.show()

# Print the Schema (the data types Spark guessed)
print("The Schema (Data Types):")
sales_df.printSchema()

### 🛑 Shutting Down
When you are finished with a Spark Application, it is best practice to shut down the SparkSession to free up the CPU and Memory resources.

In [ ]:
# Stop the engine
spark.stop()
print("Spark engine successfully stopped. Resources freed.")

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Setting up Spark looks easy here, but in the real world, environment management is a major dividing line between data roles.

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Environment Setup** | Installs SQL Server on a dedicated physical machine. | **Configures massive Spark clusters in the Cloud. Chooses instance types and configures YARN/K8s.** | Just wants a Jupyter Notebook with a working SparkSession already provided. |
| **Testing Strategy** | Tests queries on a "Dev" database. | **Develops PySpark code in `local[*]` mode on a laptop, then deploys the exact same code to a production cluster.** | Tests models on small data locally, hands code to DE to run on Big Data. |
| **Troubleshooting** | Database connection issues. | **Resolving Spark version conflicts, Java dependency errors, and cluster connectivity issues.** | Package version conflicts (`pip install` errors). |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Learns to run PySpark in `local[*]` mode (like we just did). Focuses heavily on the PySpark DataFrame syntax.
2. **Mid-Level DE:** Runs Spark on managed Cloud platforms (like Databricks or AWS EMR). Learns how to configure a remote SparkSession to securely access massive S3 buckets or Data Lakes.
3. **Senior/Platform DE:** Builds the actual infrastructure. Sets up Kubernetes clusters so that every Data Scientist in the company gets their own automated SparkSession on demand.

### 🛠️ Skills Matrix: Where we are heading
You now know how to boot up Spark and create a DataFrame! 
But wait... when you ran `sales_df = spark.read.csv(...)`, did Spark actually read the data right then? 
The answer is NO! In the next section, we will uncover the secret to Spark's speed: **Lazy Evaluation**.

## 🔑 Key Takeaways

- **Local Mode:** PySpark can run on a single laptop using `master("local[*]")`. It simulates a cluster by using your local CPU cores as Executors.
- **SparkSession:** This is the entry point for your application. It creates the Driver and allows you to interact with the engine.
- **Spark DataFrames:** The core data structure in modern Spark. They look like normal tables but are distributed under the hood.
- **Reading Data:** The `spark.read` API allows you to easily ingest data from CSV, JSON, Parquet, and more with built-in schema inference.

## 📝 Knowledge Check Quiz

**Question 1:** When initializing a SparkSession, what does `.master("local[*]")` tell the Spark engine to do?
A) Connect to an AWS cloud cluster named "local".
B) Run Spark locally on the current machine, using all available CPU cores to simulate worker nodes.
C) Download data from the local hard drive only.
D) Run the code on a single core, disabling parallel processing.

**Question 2:** Why is a Spark DataFrame different from a standard Pandas DataFrame?
A) A Spark DataFrame only holds numbers, while Pandas holds strings.
B) A Spark DataFrame is designed to be distributed across many computers in a cluster, whereas a Pandas DataFrame lives entirely in the RAM of one single computer.
C) Spark DataFrames cannot read CSV files.
D) Spark DataFrames use the MapReduce disk-writing protocol.

**Question 3:** What is the primary purpose of calling `spark.stop()` at the end of a PySpark script?
A) To delete the data you just processed.
B) To save the data to HDFS.
C) To shut down the Driver and gracefully free up the CPU and Memory resources on the cluster.
D) To uninstall the PySpark library from your computer.

---

*Answers: 1: B, 2: B, 3: C*

### 🚀 What's Next?
We have our engine running! 

Before we dive deep into complex transformations, we need to understand *how* Spark thinks. Spark doesn't execute code the same way standard Python does. In **SECTION B: HOW SPARK WORKS INTERNALLY**, we will explore the mind-bending concept of **10.4 Lazy Evaluation**. See you there!